# HanuAi ML Assessment — Task 2: EDA, Text Mining & Clustering

**Input:** `Hanuai task2 assignment.csv` (1,000 event records in the supplied file).  
**Goal:** profile data quality, explore key fields, derive interpretable issue tags from narrative text, and group similar records using TF-IDF + K-Means.

> Important: rule-based tags and unsupervised clusters are analytical aids, not verified engineering root causes. Validate them with domain experts before making product or safety decisions.

## 1. Setup and load data
Place the source CSV in the same folder as this notebook. The analysis uses pandas, matplotlib, and scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, ast
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

df = pd.read_csv("Hanuai task2 assignment.csv")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

## 2. Data quality assessment
The report keeps original columns intact. Dates are parsed into additional fields so the source values are not overwritten. Missing values in narrative fields are filled with empty strings only for text processing; missing categorical labels are not blindly imputed.

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(quality.sort_values("missing_pct", ascending=False))
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Event IDs:", df["Event id"].duplicated().sum())

for col in ["Opened date", "BUILD_DATE", "IN_USE_DATE"]:
    df[col + "_parsed"] = pd.to_datetime(df[col], errors="coerce", utc=True)
    print(col, "parse failures:", df[col].notna().sum() - df[col + "_parsed"].notna().sum())

## 3. Numeric, categorical and temporal EDA
Use event counts and distributions to identify where the records concentrate. Interpret event volume in context: the file may be a selected sample rather than a population-wide reliability dataset.

In [ ]:
print(df[["MODLYR"]].describe())
for col in ["CAUSAL_CD_DESC", "COMPLAINT_CD_DESC", "BUILD_PLANT_DESC", "MAKE", "MODEL"]:
    print("\\nTop values:", col)
    display(df[col].fillna("Missing").value_counts().head(15).to_frame("count"))

opened_year = df["Opened date_parsed"].dt.year.value_counts().sort_index()
opened_year.plot(kind="bar", figsize=(9,4), title="Events by opened year")
plt.ylabel("Event count"); plt.tight_layout(); plt.show()

## 4. Build narrative text and extract interpretable issue tags
The source has three free-text fields: `CAUSAL_VERBATIM`, `CORRECTION_VERBATIM`, and `CUSTOMER_VERBATIM`. The regex taxonomy below is transparent and easy to audit. It is intentionally multi-label because one event can mention several systems or symptoms.

In [ ]:
text_cols = ["CAUSAL_VERBATIM", "CORRECTION_VERBATIM", "CUSTOMER_VERBATIM"]
for col in text_cols:
    df[col] = df[col].fillna("").astype(str)
df["combined_text"] = df[text_cols].agg(" ".join, axis=1).str.replace(r"\\s+", " ", regex=True).str.strip()

issue_rules = {
 "Display / Black Screen": [r"black screen", r"blank screen", r"display", r"screen"],
 "Radio / Audio Failure": [r"radio", r"audio", r"no sound", r"satellite radio"],
 "Connectivity / Network": [r"wi.?fi", r"hotspot", r"internet", r"connect", r"ethernet", r"signal"],
 "Electrical / Power": [r"power", r"ignition", r"battery", r"voltage", r"ground circuit", r"fuse"],
 "Controls / Buttons": [r"button", r"knob", r"control", r"unresponsive", r"irresponsive"],
 "Software / Programming": [r"software", r"program", r"update", r"firmware", r"reset", r"sps"],
 "Communication / Data Bus": [r"communication", r"data bus", r"module", r"dtc", r"lost communication"],
 "Antenna / Satellite Signal": [r"antenna", r"satellite", r"coax", r"signal lost"],
 "Intermittent Failure": [r"intermitt", r"comes and goes", r"disconnect"]
}
def mine_tags(text):
    t = text.lower()
    return [tag for tag, patterns in issue_rules.items()
            if any(re.search(p, t) for p in patterns)]
df["Mined Issue Tags"] = df["combined_text"].apply(mine_tags)
df["Primary Issue Category"] = df["Mined Issue Tags"].apply(
    lambda tags: tags[0] if tags else "Other / Unclassified")
display(df[["Event id", "Mined Issue Tags", "Primary Issue Category"]].head(10))

## 5. Extract existing structured labels
The supplied tag columns are string representations of lists. Parse them safely with `ast.literal_eval`; do not use `eval` on data.

In [ ]:
def parse_listish(value):
    if pd.isna(value): return []
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else [str(parsed)]
    except (ValueError, SyntaxError):
        return [str(value)] if str(value).strip() else []

for col in ["Failure Component", "Failure Condition", "Trigger",
            "Additional Context", "Fix Component", "Fix Condition"]:
    df[col + "_parsed"] = df[col].apply(parse_listish)

df["Mined Component Tags"] = df["Failure Component_parsed"].apply(
    lambda xs: "; ".join(dict.fromkeys(xs)))
df["Mined Symptom Tags"] = df["Failure Condition_parsed"].apply(
    lambda xs: "; ".join(dict.fromkeys(xs)))
df["Mined Fix Tags"] = df["Fix Condition_parsed"].apply(
    lambda xs: "; ".join(dict.fromkeys(xs)))

## 6. Text mining: TF-IDF + K-Means
This unsupervised step groups records by shared vocabulary. The cluster top terms help analysts interpret each group. Cluster IDs are arbitrary and are not severity rankings.

In [ ]:
text_for_model = df["combined_text"].replace("", "no narrative text")
vectorizer = TfidfVectorizer(stop_words="english", max_features=2500,
                             ngram_range=(1,2), min_df=2)
X = vectorizer.fit_transform(text_for_model)
n_clusters = min(6, max(2, int(round(np.sqrt(len(df)/2)))))
model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df["Text Cluster"] = model.fit_predict(X)
terms = np.array(vectorizer.get_feature_names_out())
cluster_terms = {}
for cluster_id in range(n_clusters):
    top_idx = model.cluster_centers_[cluster_id].argsort()[::-1][:7]
    cluster_terms[cluster_id] = ", ".join(terms[top_idx])
df["Cluster Top Terms"] = df["Text Cluster"].map(cluster_terms)
display(df.groupby(["Text Cluster", "Cluster Top Terms"]).size()
        .reset_index(name="event_count").sort_values("event_count", ascending=False))

## 7. Summaries, export and validation
Export the original columns plus derived fields. Before operational use, manually review a stratified sample of tagged and untagged records, estimate precision/recall, and refine the taxonomy.

In [ ]:
# Category counts
category_counts = df["Primary Issue Category"].value_counts().rename_axis("Issue Category").reset_index(name="Event Count")
category_counts["Share %"] = (category_counts["Event Count"] / len(df) * 100).round(1)
display(category_counts)

# Export enriched records; keep source fields for traceability.
derived = ["combined_text", "Mined Issue Tags", "Primary Issue Category",
           "Mined Component Tags", "Mined Symptom Tags", "Mined Fix Tags",
           "Text Cluster", "Cluster Top Terms"]
df.to_csv("task2_enriched_dataset.csv", index=False, encoding="utf-8-sig")
category_counts.to_csv("task2_issue_category_summary.csv", index=False)
print("Saved task2_enriched_dataset.csv and task2_issue_category_summary.csv")

# Also create an Excel workbook with the main analysis outputs.
# These sheets make it easier for a stakeholder to review results without running Python.
cluster_summary = (df.groupby(["Text Cluster", "Cluster Top Terms"])
                   .size().reset_index(name="Event Count")
                   .sort_values("Event Count", ascending=False))
opened_year_counts = (df["Opened date_parsed"].dt.year.value_counts()
                      .sort_index().rename_axis("Opened Year")
                      .reset_index(name="Event Count"))

with pd.ExcelWriter("task2_analysis_outputs.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Enriched Data", index=False)
    quality.reset_index().rename(columns={"index": "Column"}).to_excel(
        writer, sheet_name="Data Quality", index=False)
    category_counts.to_excel(writer, sheet_name="Issue Categories", index=False)
    cluster_summary.to_excel(writer, sheet_name="Text Clusters", index=False)
    opened_year_counts.to_excel(writer, sheet_name="Opened by Year", index=False)

print("Saved task2_analysis_outputs.xlsx")


## 8. Business interpretation checklist
- Which issue categories are most common, and do they change over time?
- Do certain build plants, model years, makes or models have elevated event counts after accounting for exposure?
- Which symptoms co-occur with specific components and repair actions?
- Are repeat visits or long time-to-close measurable? This file alone may not contain enough fields to answer.
- Validate suspected root causes with engineering evidence, service bulletins, repair outcomes and vehicle exposure denominators.